<a href="https://colab.research.google.com/github/childanefh/HF_AI-assisted-interview/blob/main/Latest_Neo4j_mistral_tools_separate_conv_Markdown_Colab_ornot_separate_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description

The purpose of this script is to use Mistral AI models to support **operational debriefing** and especially the post-incident form.
the witness starts by giving facts about the event, then the system will generate pesonalised questions to clarify the situation and analyse the key **Human Factors** involved in the situation.

This script include two different ways to live the interview.
- The Collaborative model
- The Non-Collaborative model

To make it work, an access to neo4j graphdatabase is needed with the database ready to go. It is free and you will need to use the CSV file given to import the graph.


In [ ]:
!apt-get install -y portaudio19-dev
%pip install --upgrade mistralai neo4j markdown pyaudio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
portaudio19-dev is already the newest version (19.6.0-1.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import json
from neo4j import GraphDatabase
from mistralai.client import Mistral
from google.colab import userdata, output
from IPython.display import Markdown, display
import webbrowser
import os

from mistralai.client.models import AudioFormat, RealtimeTranscriptionError, RealtimeTranscriptionSessionCreated, TranscriptionStreamDone, TranscriptionStreamTextDelta
from mistralai.extra.realtime import UnknownRealtimeEvent
import io

import asyncio
from typing import AsyncIterator

from IPython.display import display, Javascript, HTML
import base64
import pyaudio
from pydub import AudioSegment

An **API key** is like a password or a digital key that lets a program (like this script) access a service or data from another system (like Mistral AI or neo4j).

Neo4j also needs its own key and url to allow an access to your personal database.

In [ ]:
api_key= userdata.get("MISTRAL_API_KEY")
neo4j_password =  userdata.get("NEO4J_P")
neo4j_user =  userdata.get("NEO4J_U")
neo4j_uri =  userdata.get("NEO4J_URI")
client = Mistral(api_key=api_key,timeout_ms=180000)

The next part is using javascript to transcribe the answer from speech to text.
It is also using a mistral agent to this extent.

In [ ]:
async def audio_chunk_generator(audio_stream, chunk_size=4000):
    """Yield chunks of audio data asynchronously."""
    audio_stream.seek(0)  # Rewind to the start of the stream
    while True:
        chunk = audio_stream.read(chunk_size)
        if not chunk:
            break
        yield chunk
        await asyncio.sleep(0)  # Yield control to the event loop

In [ ]:
async def transcribe_audio(wav_data):

    # Convert the audio data to a stream in pcm
    audio_segment = AudioSegment.from_file(io.BytesIO(wav_data), format="webm")
    audio_segment = audio_segment.set_frame_rate(16000).set_channels(1).set_sample_width(2)
    raw_audio_bytes = audio_segment.raw_data
    audio_stream = io.BytesIO(raw_audio_bytes)
    async def audio_chunks():
        async for chunk in audio_chunk_generator(audio_stream):
            yield chunk

    audio_format = AudioFormat(encoding="pcm_s16le", sample_rate=16000)

    transcribed_text = []

    try:
        async for event in client.audio.realtime.transcribe_stream(
            audio_stream=audio_chunks(),
            model="voxtral-mini-transcribe-realtime-2602",
            audio_format=audio_format,
            target_streaming_delay_ms=1000,
        ):
            #print(f"Received event: {type(event)}")
            if isinstance(event, RealtimeTranscriptionSessionCreated):
                print(f"Session created.")
            elif isinstance(event, TranscriptionStreamTextDelta):
                text = event.text
                transcribed_text.append(text)
                #print(event.text)
            elif isinstance(event, TranscriptionStreamDone):
                print("Transcription done.")
            elif isinstance(event, RealtimeTranscriptionError):
                print(f"Error: {event}")
            elif isinstance(event, UnknownRealtimeEvent):
                print(f"Unknown event: {event}")
                continue
    except KeyboardInterrupt:
        print("Stopping...")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    #print(" ".join(transcribed_text))
    return " ".join(transcribed_text)

# Run the transcription after recording

async def main_transcribe(f):
    if f :
        return await transcribe_audio(f)
    else:
        print("No audio data received.")


In [ ]:
js = Javascript("""

async function recordAudio() {
  return new Promise(async (resolve) => {
    const div = document.createElement('div');
    const audio = document.createElement('audio');
    const startButton = document.createElement('button');
    const stopButton = document.createElement('button');
    const pauseButton = document.createElement('button');

    startButton.textContent = 'Start Recording';
    stopButton.textContent = 'Stop Recording';
    pauseButton.textContent = 'Pause';

    div.appendChild(startButton);
    div.appendChild(pauseButton);
    document.body.appendChild(div);

    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream, { mimeType: 'audio/webm' });

    let chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);

    let state = "idle"; // idle | recording | paused

    // ▶️ START
    startButton.onclick = () => {
      startButton.replaceWith(stopButton);
      recorder.start();
      state = "recording";
    };

    // SPACE KEY → START
    document.addEventListener("keydown", (event) => {
      if (event.code === "Space") {
        event.preventDefault(); // prevent page scrolling
        startButton.click();
        if (state === "idle") {
          startButton.click();
        }
        else {
          pauseButton.click();
        }
      }
    });

    // ⏸️ PAUSE / RESUME toggle
    pauseButton.onclick = () => {
      if (state === "recording") {
        recorder.pause();
        pauseButton.textContent = "Resume";
        state = "paused";
      }
      else if (state === "paused") {
        recorder.resume();
        pauseButton.textContent = "Pause";
        state = "recording";
      }
    };

    // ⏹️ STOP
    stopButton.onclick = async () => {
      recorder.stop();

      recorder.onstop = async () => {
        const blob = new Blob(chunks, { type: 'audio/webm' });
        const arrBuff = await blob.arrayBuffer();

        stream.getTracks().forEach(track => track.stop());
        div.remove();

        let binaryString = '';
        let bytes = new Uint8Array(arrBuff);
        bytes.forEach(byte => binaryString += String.fromCharCode(byte));

        resolve(btoa(binaryString)); // Resolve the promise with the base64 string
      };
    };
  });
}

""")

This part is a Mistral agent equiped with a tool to access the neo4j graphdatabase and gather infomation in order to process the event with more reliability.

In [ ]:
URI = neo4j_uri
AUTH = (neo4j_user, neo4j_password)

def run_cypher_query(cypher_query):
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        records, _, _ = driver.execute_query(cypher_query,database_="neo4j")
    #return records
    result = [dict(record) for record in records]
    return str(result)

This next part is in order to describe the function and allow the agent to use it properly and efficiently.

In [ ]:
tool = [
    {
        "type": "function",
        "function": {
            "name": "run_cypher_query",
            "description": ( """Use the neo4j database by yourself by sending cypher queries, Neo4j database with the following schema:
                - Labels : "HFACS", "Vocabulary"
                - Relationships : "MAY_INVOLVE", "REPORTS_TO", "MAY_RESULT_IN", "MAY_BE_DUE", "LINKED"

                "HFACS" label has the following properties :
                  - title
                  - class
                  - sub_to

                "Vocabulary" label has the following properties :
                  - name
                  - precision
                  - link"""),
            "parameters": {
                "type": "object",
                "properties": {
                    "cypher_query": {
                        "type": "string",
                        "description": f"""The cypher query to search in the database. Exemples: MATCH (v:Vocabulary) WHERE toLower(v.name) CONTAINS "maintenance" OR toLower(v.precision) CONTAINS "maintenance" OR toLower(v.name) CONTAINS "repair" OR toLower(v.precision) CONTAINS "repair" OR toLower(v.name) CONTAINS "technical" OR toLower(v.precision) CONTAINS "technical" OR toLower(v.name) CONTAINS "equipment" OR toLower(v.precision) CONTAINS "equipment" OR toLower(v.name) CONTAINS "failure" OR toLower(v.precision) CONTAINS "failure" MATCH (v)-[r:REPORTS_TO|MAY_INVOLVE|MAY_BE_DUE|MAY_RESULT_IN|LINKED]->(h:HFACS) MATCH (v)-[r2:REPORTS_TO|MAY_INVOLVE|MAY_BE_DUE|MAY_RESULT_IN|LINKED]->(v2:Vocabulary) RETURN v.name, v.precision, h.title, h.class, h.sub_to, type(r) AS relationship, v2.name, v2.precision, type(r2) AS relationship2;"""
                    }
                },
                "required": ["cypher_query"],
            },
        },
    }
]

In [ ]:
def receive_the_query(input_, ToolCall):
    prompt = f"""You are a human factor expert agent interacting with a Neo4j database with the following schema:

    - Labels : "HFACS", "Vocabulary"
    - Relationships : "MAY_INVOLVE", "REPORTS_TO", "MAY_RESULT_IN", "MAY_BE_DUE", "LINKED"

    19 "HFACS" label with the following properties :
    - title
    - class
    - sub_to

    118 "Vocabulary" label with the following properties :
    - name
    - precision
    - link

    The tool function will let you search for relevant information in the database using relevant vocabulary from {input_} or vocabulary with the the same meaning or the same roots.
    You dispose of a single tool function called "run_cypher_query" that will let you search for relevant information in the database. You may use "CONTAINS" and "ToLower()" in the queries.
    It has one parameter "cypher_query" that is a list of strings. you have to search for every piece of information that could help understand how human factors are involved in {input_}. Be careful to properly use cypher queries.
    {ToolCall} is the fruit of your previous research. It is to avoid searching twice the same word and it allows you to try searching for the neighbours of a "Vocabulary" node you found in the database.

    You have then to return the fruit of your research in the database.


    Your answer:

    """

    chat_response = client.chat.complete(
        model= "mistral-large-latest",
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
        ],
        tools= tool
    )


    if chat_response.choices[0].message.tool_calls:
        for tool_call in chat_response.choices[0].message.tool_calls:
            if tool_call.function.name == "run_cypher_query":
                parsed_arguments = {} # Initialize parsed_arguments
                # Parse the arguments if they are a string
                if isinstance(tool_call.function.arguments, str):
                    try:
                        parsed_arguments = json.loads(tool_call.function.arguments)
                    except json.JSONDecodeError:
                        print(f"Error decoding JSON arguments: {tool_call.function.arguments}")
                        # Skip this tool call if arguments are malformed
                        continue
                else:
                    parsed_arguments = tool_call.function.arguments

                # Check if 'cypher_query' key exists in parsed_arguments
                if "cypher_query" in parsed_arguments:
                    try:
                        result = run_cypher_query(parsed_arguments["cypher_query"])
                        second_response = client.chat.complete(
                            model="mistral-large-latest",
                            messages=[
                                {"role": "user", "content": "Please use the database"},
                                {"role": "assistant", "content": None, "tool_calls": [tool_call]},
                                {"role": "tool", "content": result, "tool_call_id": tool_call.id}
                            ]
                        )
                        ToolCall.write(f"""<span style="color: green">ToolCall:</span> {second_response.choices[0].message.content}

""")
                    except Exception as e:
                        ToolCall.write(f"""<span style="color: darkgreen">Error:</span> {e}

""")
                        # Handle error in query execution if needed
                else:
                    ToolCall.write(f"""<span style="color: lightgreen">Error:</span> 'cypher_query' key not found in parsed arguments

""")
                    # If 'cypher_query' is missing, the tool call cannot proceed as expected.
                    # We just skip processing this specific tool call.

    return ToolCall

This agent is made to summarise the user's answer. It is only using a smaller model.

In [ ]:
def SumUP_2(input_):
    prompt = f"""

    Role:
        You are a human factor expert agent interacting with a user that has lived an aeronotical incident:

    Input:
        You received the following input from the user: {input_}

    Task:
        return a summary of the whole situation:

    Output Format:
        "Summary of the situation":
            use the user's input ({input_}) to describe the situation. Be clear, concise and aerate your answer.

    Rules:
        Do not add anything outside the summary.
        Base you summary on {input_}
        Keep the summary clear, concise, and non-technical (avoid jargon). except if it has been used by the user.
        Do not use the Markdown key to make lines between parts: "---"


    """

    chat_response = client.chat.complete(
        model= "mistral-small-latest",
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    return chat_response.choices[0].message.content


This next agent is made to partially analyse the situation returning an analyse in two parts: The first part concern the Human Factors that are involved in the event whereas the second part concern Human Factors that may be involved in the situation and need a clarification trough some questions.
The analyse is based on the HFACS.

In [ ]:
def Analyse2(ToolCall, Messages, Analyses, analyse_opinion = ""):
    prompt = f"""
    Role:
        You are a human factors expert analyzing an aeronautical incident described by a user. Your task is to identify and explain the human factors involved using the HFACS framework and associated vocabulary.

    Inputs:
        You have access to the following information:
            Database information: {ToolCall}.
            Exchange history with the user: {Messages} (user’s anecdote and previous interactions).
            Your previous analyses: {Analyses} (past analyses you’ve generated).
            User’s opinion on your last analysis: {analyse_opinion}. May be an empty str: ""

    HFACS Framework Reference:
        Use the following HFACS levels and categories to classify and explain the human factors in the incident:

            HFACS Level 1: Unsafe Acts
                Errors (unintentional behaviors):
                    Skill-Based Errors
                    Decision Errors
                    Perceptual Errors
                Violations (intentional disregard for rules):
                    Routine Violations
                    Exceptional Violations

            HFACS Level 2: Preconditions for Unsafe Acts
                Environmental Factors:
                    Physical Environment
                    Technological Environment
                Condition of Operators:
                    Adverse Mental State
                    Adverse Physiological State
                    Physical/Mental Limitations
                Personnel Factors:
                    Crew Resource Management
                    Personal Readiness

            HFACS Level 3: Unsafe Supervision
                Inadequate Supervision
                Plan Inappropriate Operation
                Fail to Correct Known Problem
                Supervisory Violation

            HFACS Level 4: Organisational Influences
                Resource Management
                Organisational Climate
                Operational Process


    Task:
        Analyze the human factors involved in the incident based on:
            {Messages} (user’s anecdote),
            {ToolCall} (HFACS framework and vocabulary),
            {Analyses} (previous analyses),
            {analyse_opinion} (user’s feedback).

    Output Format:
        1- Key Human Factors Involved:
            - [Your concise analysis here]
            Associate the **HFACS categories/subcategories** from the framework above with the real situation described in `{Messages}` and data found in {ToolCall} to better understand the situation.
            Base you analysis on the previous **"1- Key Human Factors Involved"** section in `{Analyses}` and the possible opinion about it in `{analyse_opinion}`.
            Be **clear, concise and aerate your answer**.

        2- Possible Human Factors Involved:
            - [Your concise analysis here]
            Identify **additional possible human factors** using the HFACS framework, `{Messages}` and data / Vocabulary nodes found in {ToolCall} to better understand the situation.
            Base you analysis on the previous **"2- Possible Human Factors Involved"** section in `{Analyses}` and the possible opinion about it in `{analyse_opinion}`. This part may decrease after each loop.
            If all possible factors have been covered in **"1- Key Human Factors Involved"**, return `None`.


    Rules:
        Use only the HFACS terms/categories provided above for classification.
        Do not add any additional text or explanations outside the two-part structure.
        Base your analysis only on the provided inputs ({ToolCall}, {Messages}, {Analyses}, {analyse_opinion}).
        Ensure your explanations are clear, concise, and non-technical (avoid jargon), except if it is used by the user.
        If no further factors are identified, return only "None" for part 2.
        Do not use the Markdown key to make lines between parts: "---"

    Example Output:
        1- Key Human Factors Involved:
            - The incident involved *decision errors* (HFACS Level 1) due to inadequate risk assessment during poor weather (*physical environment*, HFACS Level 2).
            - The user also described *adverse mental state* (stress) and *crew resource management* issues (poor coordination).


        2- Possible Human Factors Involved:
            - Possible *supervisory violation* (HFACS Level 3) if the risky operation was condoned by management.
            - *Organisational climate* (HFACS Level 4) may have contributed if safety culture was lax.


    """

    chat_response = client.chat.complete(
        model= "mistral-medium-latest",
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    return chat_response.choices[0].message.content

This agent is basing a question to clarify the event and especially question the involvment of Human Factors of the second part of the partial analysis given by the previous agent.

In [ ]:
def ask_question2(ToolCall, Messages, Analyses, Questions, analyse_opinion = ""):
    prompt = f"""

    You are a human factors expert analyzing an aeronautical incident shared by a user. You have access to the following information:
        Database information: {ToolCall}
        Exchange history with the user: {Messages}
        Your previous analyses: {Analyses}
        Questions you’ve already asked: {Questions}
        User’s opinion on your last analysis: {analyse_opinion}

    Your Task:
        Analyze the human factors involved in the incident using:
            {Analyses}, {analyse_opinion}, {Messages}, and {ToolCall}.
            The HFACS framework and its associated Vocabulary nodes.

        Ask the user a single, open-ended question to:
            Help them develop an aspect of their anecdote that clarifies or resolves uncertainties in the "2- Possible Human Factors involved" section of the end of {Analyses}.
            Use simple, non-technical language (avoid jargon), except if it is used by the user.
            You may provide **2 to 4 specific examples** from {Messages} to guide their response. (It is not mandatory)
            Avoid repeating questions from {Questions} that explore the same topic.

        Constraints:
            Your question must be under 500 characters.
            Do not explain why you’re asking the question.
            Return Only "Thank you, it will be enough for today" if:
                The "2- Possible Human Factors involved" section in {Analyses} is empty or equal to "None", or
                You determine no further relevant questions are needed.


        Output Format:
            Return only the question or the closing statement. No additional text.


    Example Outputs:
        Valid question:
            "You mentioned feeling rushed during pre-flight checks. Can you describe what specifically made you feel hurried, or if there were any distractions at that time? For example, were there time pressures, interruptions, or unclear procedures?"
            "Did you feel any extra pressure or mental block related to the situation, or was it really just a purely mechanical and cognitive confusion?"
            "When the porpoising happened, You said that you acted quickly and appropriately—but since the event was unexpected, do you think that the aircraft’s design or handling characteristics might have played a role in making the oscillations harder to control?
                For example:
                    - Did this aircraft feel more prone to porpoising in gusty conditions compared to other aircraft you’ve flown?
                    - Were there any past discussions about how its aerodynamics or systems (like the nosewheel damping) behave during firm landings?"


    Closing statement:
        "Thank you, it will be enough for today."

    """

    chat_response = client.chat.complete(
        model= "mistral-medium-latest",
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    return chat_response.choices[0].message.content

Finally, This agent is processing the whole exchange to return a detailled analyse of the event and classify it using HFACS as basis.

In [ ]:
def Classify2(Messages, Analyses, added_input_):
    prompt = f"""

    Role:
        You are a human factors expert analyzing an aeronautical incident described by a user. Your task is to provide a detailed summary of the situation and identify the key and secondary human factors involved, based on the exchange history and previous analyses.

    Inputs:
        You have access to the following information:
            Exchange history with the user: {Messages}
            Your previous analyses: {Analyses}
            Additional input: {added_input_} (optional new information to adapt the analysis).

    HFACS Framework Reference:
        Use the following HFACS levels and categories to classify and explain the human factors in the incident:

            HFACS Level 1: Unsafe Acts
                Errors (unintentional behaviors):
                    Skill-Based Errors
                    Decision Errors
                    Perceptual Errors
                Violations (intentional disregard for rules):
                    Routine Violations
                    Exceptional Violations

            HFACS Level 2: Preconditions for Unsafe Acts
                Environmental Factors:
                    Physical Environment
                    Technological Environment
                Condition of Operators:
                    Adverse Mental State
                    Adverse Physiological State
                    Physical/Mental Limitations
                Personnel Factors:
                    Crew Resource Management
                    Personal Readiness

            HFACS Level 3: Unsafe Supervision
                Inadequate Supervision
                Plan Inappropriate Operation
                Fail to Correct Known Problem
                Supervisory Violation

            HFACS Level 4: Organisational Influences
                Resource Management
                Organisational Climate
                Operational Process

    Task:
        Return a two-part answer based on {Messages} and {Analyses}, adapting to {added_input_} if necessary.

    Output Format:
        1- Summary of the Situation:
            Provide a detailed summary of the anecdote using the exchange in `{Messages}`.

        2- Key Human Factors Involved:
            Use the **last analysis in `{Analyses}`** and adapt it to `{added_input_}` if needed.
            Then, list **up to 7 most important HFACS factors** involved in this situation, with clear, concise and aerate explanations based on `{Messages}`.
            If fewer than 5 factors are relevant, return only those.
            Order them by importance; putting in first place the HFACS that needs to be studied.
            Example:
                - "Time Pressure (HFACS Level 2): The user mentioned feeling rushed due to delayed departure..."
                - "Miscommunication (HFACS Level 2): Poor coordination with ATC led to confusion..."

    Rules:
        Do not add anything outside the two-part structure.
        Base your analysis only on {Messages}, {Analyses}, {added_input_}. Ignore any other sources.
        Adapt the analysis to {added_input_} only if it is provided and relevant.
        Avoid ordering the analysis in terms of HFACS framework / levels
        Keep explanations clear, concise, and non-technical (avoid jargon). except if it has been used by the user.
        Do not use the Markdown key to make lines between parts: "---"


    Example Output:
        1- Summary of the Situation:
            The user described a situation where they experienced a near-miss during takeoff due to a miscommunication with the co-pilot. The user felt rushed because of a delayed departure and skipped a critical checklist step. Weather conditions were also poor, adding to the stress.


        2- Key Human Factors Involved:
            - Adverse Mental State (HFACS Level 2): time pressure : The user mentioned feeling rushed due to a delayed departure, leading to skipped steps.
            - Crew Resource Management (HFACS Level 2): miscommunication : Poor coordination with the co-pilot resulted in confusion during takeoff.
            - Adverse Environmental Conditions (HFACS Level 2): adverse weather : Poor weather conditions contributed to the stress and complexity of the situation.


    """

    chat_response = client.chat.complete(
        model= "mistral-medium-latest",
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
        ]
    )

    return chat_response.choices[0].message.content

##Collaborative model

This model is made to counter AI hallucinations and over-interpretations by asking the user to review the summary and the analyse in each loop.


To make it work, you have to open the files and then start the code. You always have to dowload the files after each interview. Otherwise, you will loose everything.
Moreover, to open the files, it will overwrite ancient files so you have to dowload them before starting a new session.

If you need to, you can also use old files to to resume the interview where you stopped it.  



In [ ]:
open("ToolCall.md", "w", encoding="utf-8")
open("Messages.md", "w", encoding="utf-8")
open("Questions.md", "w", encoding="utf-8")
open("Analyses.md", "w", encoding="utf-8")
open("Exchange.md", "w", encoding="utf-8")

<_io.TextIOWrapper name='Exchange.md' mode='w' encoding='utf-8'>

In [ ]:
with open("ToolCall.md", "r+", encoding="utf-8") as ToolCall:
    with open("Messages.md", "r+", encoding="utf-8") as Messages:
        with open("Questions.md", "r+", encoding="utf-8") as Questions:
            with open("Analyses.md", "r+", encoding="utf-8") as Analyses:
                with open ("Exchange.md", "r+", encoding="utf-8") as Exchange:
                    summary_opinion = ""
                    analyse_opinion = ""
                    i = 0
                    print ('Can you describe the situation and give me the facts about it.')
                    while True:
                        print ("Answer: ")
                        display(js)
                        recorded_audio_output = output.eval_js('recordAudio({})')
                        wav_data = base64.b64decode(recorded_audio_output)
                        with open(f'audio_ans{i}.webm', 'wb') as f:
                            f.write(wav_data)
                        input_ = await main_transcribe(wav_data)
                        Messages.seek(0,2)
                        Messages.write( f"""<span style="color: red">USER: </span> {input_}

""")
                        Messages.seek(0)

                        Exchange.seek(0,2)
                        Exchange.write( f"""<span style="color: red">USER: </span> {input_}

""")
                        Exchange.seek(0)

                        ToolCall.seek(0,2)
                        receive_the_query(input_, ToolCall)
                        ToolCall.seek(0)

                        summary = SumUP_2(input_)
                        display(Markdown(summary))
                        Messages.seek(0,2)
                        Messages.write( f"""<span style="color: blue">ASSISTANT: </span>

{summary}

""")
                        Messages.seek(0)

                        Exchange.seek(0,2)
                        Exchange.write( f"""<span style="color: blue">ASSISTANT: </span>

{summary}

""")
                        Exchange.seek(0)

                        print ("Do you have anything to say on this summary?")
                        display(js)
                        recorded_audio_output = output.eval_js('recordAudio({})')
                        wav_data = base64.b64decode(recorded_audio_output)
                        with open(f'audio_sum{i}.webm', 'wb') as f:
                            f.write(wav_data)
                        summary_opinion = await main_transcribe(wav_data)
                        Messages.seek(0,2)
                        Messages.write( f"""<span style="color: red">USER: </span> {summary_opinion}

""")
                        Messages.seek(0)

                        Exchange.seek(0,2)
                        Exchange.write( f"""<span style="color: red">USER: </span> {summary_opinion}

""")
                        Exchange.seek(0)

                        analyse = Analyse2(ToolCall.read(), Messages.read(), Analyses.read(), analyse_opinion)
                        display(Markdown(analyse))
                        Analyses.seek(0,2)
                        Analyses.write( f"""<span style="color: blue">ANALYSIS: </span>

{analyse}

""")
                        Analyses.seek(0)

                        Exchange.seek(0,2)
                        Exchange.write( f"""<span style="color: blue">ANALYSIS: </span>

{analyse}

""")
                        Exchange.seek(0)

                        print ("Do you have anything to say on this analysis?")
                        display(js)
                        recorded_audio_output = output.eval_js('recordAudio({})')
                        wav_data = base64.b64decode(recorded_audio_output)
                        with open(f'audio_ana{i}.webm', 'wb') as f:
                            f.write(wav_data)
                        analyse_opinion = await main_transcribe(wav_data)
                        Analyses.write( f"""<span style="color: red">USER: </span> {analyse_opinion}

""")
                        Analyses.seek(0)

                        Exchange.seek(0,2)
                        Exchange.write( f"""<span style="color: red">USER: </span> {analyse_opinion}

""")
                        Exchange.seek(0)

                        question = ask_question2(ToolCall.read(), Messages.read(), Analyses.read(), Questions.read(), analyse_opinion)
                        display(Markdown(question))
                        Questions.seek(0,2)
                        Questions.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                        Questions.seek(0)

                        Messages.seek(0,2)
                        Messages.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                        Messages.seek(0)

                        Exchange.seek(0,2)
                        Exchange.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                        Exchange.seek(0)

                        if question == "Thank you, it will be enough for today":
                            break

                        action_ = input("Action: ")
                        while True:
                            if action_ == "exit" or action_ == "exit " or action_ == "Exit" or action_ == "Exit " or action_ == "EXIT" :
                                break
                            elif action_ == "next" or action_ == "next " or action_ == "Next" or action_ == "Next " or action_ == "NEXT":
                                question = ask_question2(ToolCall.read(), Messages.read(), Analyses.read(), Questions.read(), analyse_opinion)
                                display(Markdown(question))
                                Questions.seek(0,2)
                                Questions.write( f"""<span style="color: darkblue">ASSISTANT: </span>

{question}

""")
                                Questions.seek(0)

                                Messages.seek(0,2)
                                Messages.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                                Messages.seek(0)

                                Exchange.seek(0,2)
                                Exchange.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                                Exchange.seek(0)

                                action_ = input("Action: ")
                            else : break

                        if action_ == "exit" or action_ == "exit " or action_ == "Exit" or action_ == "Exit " or action_ == "EXIT" :
                            break

                        i += 1

                    print ("Do you have something to add ?")
                    display(js)
                    recorded_audio_output = output.eval_js('recordAudio({})')
                    wav_data = base64.b64decode(recorded_audio_output)
                    with open(f'audio_add{i+1}.webm', 'wb') as f:
                        f.write(wav_data)
                    added_input_ = await main_transcribe(wav_data)
                    Messages.seek(0,2)
                    Messages.write( f"""<span style="color: red">USER: </span> {added_input_}

""")
                    Messages.seek(0)

                    Exchange.seek(0,2)
                    Exchange.write( f"""<span style="color: red">USER: </span> {added_input_}

""")
                    Exchange.seek(0)

                    classification = Classify2(Messages.read(), Analyses.read(), added_input_)
                    display(Markdown(classification))
                    Messages.seek(0,2)
                    Messages.write( f"""<span style="color: yellow">CLASSIFICATION: </span>

{classification}

""")
                    Messages.seek(0)

                    Exchange.seek(0,2)
                    Exchange.write( f"""<span style="color: yellow">CLASSIFICATION: </span>

{classification}

""")
                    Exchange.seek(0)

                    Analyses.seek(0,2)
                    Analyses.write( f"""<span style="color: yellow">CLASSIFICATION: </span>

{classification}

""")
                    Analyses.seek(0)

Answer: 


<IPython.core.display.Javascript object>

##Non Collaborative model

This model is made to mimic a real interview between two people.

To make it work, you have to open the files and then start the code. You always have to dowload the files after each interview. Otherwise, you will loose everything.
Moreover, to open the files, it will overwrite ancient files so you have to dowload them before starting a new session.

If you need to, you can also use old files to to resume the interview where you stopped it.  

In [ ]:
open("ToolCall.md", "w", encoding="utf-8")
open("Messages.md", "w", encoding="utf-8")
open("Questions.md", "w", encoding="utf-8")
open("Analyses.md", "w", encoding="utf-8")

<_io.TextIOWrapper name='Analyses.md' mode='w' encoding='utf-8'>

In [ ]:
with open("ToolCall.md", "r+", encoding="utf-8") as ToolCall:
    with open("Messages.md", "r+", encoding="utf-8") as Messages:
        with open("Questions.md", "r+", encoding="utf-8") as Questions:
            with open("Analyses.md", "r+", encoding="utf-8") as Analyses:
                i = 0
                print ('Can you describe the situation and give me the facts about it.')
                while True:
                    print ("Answer: ")
                    display(js)
                    recorded_audio_output = output.eval_js('recordAudio({})')
                    wav_data = base64.b64decode(recorded_audio_output)
                    with open(f'audio{i}.webm', 'wb') as f:
                        f.write(wav_data)
                    input_ = await main_transcribe(wav_data)
                    Messages.seek(0, 2)
                    Messages.write( f"""<span style="color: red">USER: </span> {input_}


""")
                    Messages.seek(0)

                    ToolCall.seek(0,2)
                    receive_the_query(input_, ToolCall)
                    ToolCall.seek(0)

                    summary = SumUP_2(input_)
                    Messages.seek(0, 2)
                    Messages.write( f"""<span style="color: blue">ASSISTANT: </span>

{summary}

""")
                    Messages.seek(0)

                    analyse = Analyse2(ToolCall.read(), Messages.read(), Analyses.read())
                    Analyses.seek(0, 2)
                    Analyses.write( f"""<span style="color: blue">ANALYSIS: </span>

{analyse}

""")
                    Analyses.seek(0)

                    question = ask_question2(ToolCall.read(), Messages.read(), Analyses.read(), Questions.read())
                    display(Markdown(question))
                    Questions.seek(0, 2)
                    Questions.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                    Questions.seek(0)

                    Messages.seek(0, 2)
                    Messages.write( f"""<span style="color: blue">ASSISTANT: </span>

{question}

""")
                    Messages.seek(0)

                    if question == "Thank you, it will be enough for today":
                        break

                    action_ = input("Action: ")
                    while True:
                        if action_ == "exit" or action_ == "exit " or action_ == "Exit" or action_ == "Exit " or action_ == "EXIT" :
                            break
                        elif action_ == "next" or action_ == "next " or action_ == "Next" or action_ == "Next " or action_ == "NEXT":
                            question = ask_question2(ToolCall.read(), Messages.read(), Analyses.read(), Questions.read())
                            display(Markdown(question))
                            Questions.seek(0, 2)
                            Questions.write( f"""<span style="color: darkblue">ASSISTANT: </span>

{question}

""")
                            Questions.seek(0)

                            action_ = input("Action: ")
                        else : break

                    if action_ == "exit" or action_ == "exit " or action_ == "Exit" or action_ == "Exit " or action_ == "EXIT" :
                        break

                    i += 1


                print ("Do you have something to add ?")
                display(js)
                recorded_audio_output = output.eval_js('recordAudio({})')
                wav_data = base64.b64decode(recorded_audio_output)
                with open(f'audio{i+1}.webm', 'wb') as f:
                    f.write(wav_data)
                added_input_ = await main_transcribe(wav_data)
                Messages.seek(0, 2)
                Messages.write( f"""<span style="color: red">USER: </span> {added_input_}

""")
                Messages.seek(0)

                classification = Classify2(Messages.read(), Analyses.read(), added_input_)
                display(Markdown(classification))
                Messages.seek(0, 2)
                Messages.write( f"""<span style="color: yellow">CLASSIFICATION: </span>

{classification}

""")
                Messages.seek(0)
                Analyses.seek(0,2)
                Analyses.write( f"""<span style="color: yellow">CLASSIFICATION: </span>

{classification}

""")
                Analyses.seek(0)

Answer: 


<IPython.core.display.Javascript object>

Session created.
Transcription done.


During the turn toward the airfield, you mentioned the glide computer didn’t account for obstacles like trees. Did you notice any other limitations with the aircraft’s instruments or systems that made it harder to judge your speed, altitude, or energy state in that moment?

For example:
- Were there any warnings or cues (e.g., stall warnings, airspeed fluctuations) that you expected but didn’t get?
- Did the turn feel different from what you’d practiced—like the aircraft responding slower or more abruptly than usual?
- Were you relying on a specific instrument (e.g., variometer, altimeter) that might have given misleading or delayed feedback?

Action: go
Answer: 


...

After the incident, did you ever reflect on whether the contest’s structure—like how tasks were assigned, how scoring worked, or how safety briefings were handled—might have unintentionally pushed pilots toward riskier decisions?

For example:
- Did the way points were awarded (e.g., higher rewards for airfield landings) make alternatives like the paddock feel less acceptable?
- Were there moments where you or others felt pressured to "perform" rather than prioritize safety?
- Did the lack of shared terrain/wind briefings make it harder to judge risks as a group?

Action: exit
Do you have something to add ?


<IPython.core.display.Javascript object>

Session created.
Transcription done.


1- Summary of the Situation:
In 2006, during a gliding contest in Matamata, New Zealand, the pilot—unfamiliar with the area—attempted to reach a turn point 50 km away but struggled due to lack of local knowledge. After abandoning the task, they decided to return to the airfield, climbing to 2,000 feet and relying solely on an outdated glide computer that indicated they could reach the airfield. The computer did not account for real-time wind changes or obstacles like trees, and the pilot did not monitor other instruments or airspeed closely.

Approaching the airfield, the pilot encountered trees blocking the path and lacked the speed to clear them. While attempting to turn toward an alternate paddock, a 10-knot headwind (now a tailwind) caused the glider to stall and crash into a silage pit. The pilot survived with minor injuries but acknowledged fixating on reaching the airfield rather than exploring other landing options. They later learned a suitable paddock existed nearby but was unaware of it due to unfamiliarity with the terrain.

The pilot admitted over-relying on the glide computer without fully understanding its limitations, particularly regarding wind calculations. Pre-contest briefings did not cover emergency landing options or terrain challenges in detail, and the pilot did not seek advice, assuming such knowledge was inherent. The incident led to increased caution about flying low in unfamiliar areas.

---

2- Key Human Factors Involved:
- **Over-reliance on automation (glide computer)**: The pilot placed blind trust in an outdated glide computer that may not have updated wind calculations in real-time. This led to an incorrect assumption that the airfield was reachable, despite changing conditions (e.g., headwind becoming a tailwind). The pilot admitted not questioning the computer’s output, even when visual cues (trees, lack of speed) contradicted it.

- **Fixation (tunnel vision)**: The pilot became fixated on reaching the airfield as the sole landing option, ignoring alternative paddocks nearby. This mental tunnel vision prevented them from adapting to obstacles (trees) or considering safer alternatives, even when the original plan became unfeasible.

- **Lack of situational awareness**: The pilot failed to monitor airspeed, altitude, or other instruments closely, focusing instead on the ground and landing spots. This lack of awareness of the glider’s energy state (e.g., insufficient speed during the turn) directly contributed to the stall and crash.

- **Unfamiliarity with terrain**: Flying in an unknown area, the pilot lacked pre-planned emergency landing options and did not scout suitable paddocks during the flight. This unfamiliarity, combined with the assumption that "many paddocks" meant landing would always be possible, led to poor decision-making under stress.

- **Inadequate pre-flight preparation/briefing**: Emergency landing options and terrain challenges were not thoroughly discussed in pre-flight briefings. The pilot assumed such knowledge was inherent ("pilots should just know") and did not seek advice, leaving them unprepared for the actual conditions encountered.

- **Insufficient instrument knowledge**: The pilot admitted not fully understanding the glide computer’s limitations, particularly its inability to update wind calculations dynamically. This knowledge gap contributed to over-trust in the system and a failure to cross-check its output with other cues (e.g., wind direction, airspeed).

- **Complacency toward light wind conditions**: The pilot dismissed the 10-knot wind as "pretty light" and did not account for its impact during the turn, where it became a critical tailwind. This underestimation of environmental factors played a key role in the loss of control.